# EcoData · S7 — Ingesta de eventos de sensores en tiempo real
**Lanzeloth Vargas Marichi** · Proyecto Sello EcoData · Big Data, UPeU

Patrón de la guía: `dispositivo → puente → Kafka → consumer con validación de esquema y de rango físico`.
En EcoData:

| Guía S7 | EcoData |
|---|---|
| Dispositivo (ESP32) | Estación virtual de Juliaca en Open-Meteo (PM2.5, PM10, temperatura, humedad, viento) |
| Puente MQTT → Kafka | **`puente-openmeteo`**: llama a la API y publica un evento por hora y variable |
| Topic `atmos-eventos` | **`ecodata-s07-lecturas`** (3 particiones) — aparte de `atmos-aire`/`atmos-clima` para no alterar Grafana |
| `consumer_sensores.py` | **`validador-ecodata`**: separa `invalid` (forma) de `alerta_rango` (valor imposible) |

> ⚠️ **Transparencia sobre la fuente.** Open-Meteo entrega datos **modelados** (CAMS ≈ 45 km), no lecturas de un sensor físico propio. El patrón de ingesta es el mismo, pero la rúbrica de S7 pide un sensor real: esto se declara en el informe y la fuente puede reemplazarse por un sensor físico sin tocar el contrato ni el validador (solo cambia el puente).

Ejecuta las secciones en orden. **No repitas** la sección de publicación (duplica eventos).

## 0. Preparación

In [1]:
import sys, json, time
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import aire_comun as aire, clima_comun as clima
import openmeteo_a_kafka as pub_aire, openmeteo_clima_a_kafka as pub_clima
from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP, TOPIC = "kafka:9092", "ecodata-s07-lecturas"
def log(**kv): print(" ".join(f"{k}={v}" for k, v in kv.items()))

adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
try: adm.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=1)]); print("topic creado:", TOPIC)
except TopicAlreadyExistsError: print("el topic ya existía:", TOPIC)
adm.close()

topic creado: ecodata-s07-lecturas


## 1. Contrato de evento propio (4.1, punto 2)
Dos tipos de lectura, con campos, tipos y **unidades explícitas**. Es el mismo contrato que usa Spark (`schemas/atmos-aire-v1`, `atmos-clima-v1`), ahora validado por un consumidor independiente.

| `tipoEvento` | Campo | Tipo | Unidad | Rango físico plausible (alerta si sale) |
|---|---|---|---|---|
| `aire.lectura` | `pm25` | número | µg/m³ | 0 – 500 |
| | `pm10` | número | µg/m³ | 0 – 1000, y **pm10 ≥ pm25** (PM10 incluye a PM2.5) |
| `clima.lectura` | `temp` | número | °C a 2 m | −30 – 35 |
| | `hum` | número | % | 0 – 100 |
| | `viento` | número | unidad de la API (km/h) a 10 m | 0 – 200 |
| ambos | `sensorId` | string | — | no vacío; es la **key** |
| | `origen` | string | — | `openmeteo-cams`, `openmeteo-clima`, `prueba-falla` |
| | `timestamp` | entero | ms, UTC | hora del dato |

**Dos niveles de validación (2.2 de la guía):** (1) *esquema*: campos, tipos, `tipoEvento` → si falla es `invalid`; (2) *rango físico*: la forma es correcta pero el valor es imposible → `alerta_rango` (o `alerta_coherencia` si pm10 < pm25).

## 2. Rangos con base en datos reales
Los rangos no se inventan: se contrastan con lo realmente observado en la API para comprobar que los contienen con margen.

In [2]:
sa = pub_aire.desde_api(); sc = pub_clima.desde_api()
obs = pd.DataFrame({
    "variable": ["pm25", "pm10", "temp", "hum", "viento"],
    "min_observado": [min(x[1] for x in sa), min(x[2] for x in sa), min(x[1] for x in sc), min(x[2] for x in sc), min(x[3] for x in sc)],
    "max_observado": [max(x[1] for x in sa), max(x[2] for x in sa), max(x[1] for x in sc), max(x[2] for x in sc), max(x[3] for x in sc)],
    "rango_plausible": ["0–500", "0–1000", "−30–35", "0–100", "0–200"]})
print(f"{len(sa)} horas de aire y {len(sc)} de clima leídas de la API")
obs

50 horas de aire y 50 de clima leídas de la API


,variable,min_observado,max_observado,rango_plausible
0,pm25,0.7,8.4,0–500
1,pm10,1.1,18.7,0–1000
2,temp,2.0,19.8,−30–35
3,hum,19.0,99.0,0–100
4,viento,0.6,21.0,0–200


## 3. Puente `puente-openmeteo` (3.7–3.8)
Convierte cada hora de la API en un evento del contrato y lo publica con `acks=all`. El **sensor no habla Kafka**: el puente es quien traduce (pregunta de defensa 1).

In [3]:
RANGOS = {"aire.lectura": {"pm25": (0, 500), "pm10": (0, 1000)},
          "clima.lectura": {"temp": (-30, 35), "hum": (0, 100), "viento": (0, 200)}}
CAMPOS = {"aire.lectura": {"sensorId": str, "pm25": (int, float), "pm10": (int, float), "origen": str, "timestamp": int},
          "clima.lectura": {"sensorId": str, "temp": (int, float), "hum": (int, float), "viento": (int, float), "origen": str, "timestamp": int}}

prod = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all", key_serializer=str.encode,
                     value_serializer=lambda v: v if isinstance(v, bytes) else json.dumps(v).encode())
def enviar(ev, key, verbose=False):
    f = prod.send(TOPIC, key=key, value=ev)
    if verbose:
        m = f.get(timeout=10)
        log(service="puente-openmeteo", component="producer", topic=m.topic, partition=m.partition, offset=m.offset,
            eventType=ev["tipoEvento"] if isinstance(ev, dict) else "RAW", sensorId=key, status="published")

ev_aire  = [aire.evento(t, a, b, "openmeteo-cams") for t, a, b in sa]
ev_clima = [clima.evento(*fila, "openmeteo-clima") for fila in sc]
for i, e in enumerate(ev_aire):  enviar(e, e["sensorId"], verbose=i < 3)     # ¡UNA sola vez!
for i, e in enumerate(ev_clima): enviar(e, e["sensorId"], verbose=i < 3)
prod.flush(); print(f"publicados {len(ev_aire)} eventos de aire y {len(ev_clima)} de clima en {TOPIC}")

service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=1 offset=0 eventType=aire.lectura sensorId=openmeteo-cams-juliaca status=published
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=1 offset=1 eventType=aire.lectura sensorId=openmeteo-cams-juliaca status=published
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=1 offset=2 eventType=aire.lectura sensorId=openmeteo-cams-juliaca status=published
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=2 offset=0 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=2 offset=1 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=2 offset=2 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
publicados

## 4. Fallas provocadas (4.1, evidencia de rango)
La fuente modelada **no produce valores imposibles**, así que, para demostrar que el validador funciona, se inyectan cinco lecturas defectuosas, marcadas `origen = prueba-falla` (queda declarado en el informe: son fallas simuladas de un sensor, no datos reales).

In [4]:
t0 = int(time.time() * 1000)
base_a = aire.evento(t0, 12.0, 20.0, "prueba-falla");  base_c = clima.evento(t0, 5.0, 40.0, 10.0, "prueba-falla")
fallas = {
    "pm25 absurdo (850 µg/m³)":        {**base_a, "pm25": 850.0},
    "temperatura imposible (-80 °C)":  {**base_c, "temp": -80.0},
    "pm10 menor que pm25":             {**base_a, "pm25": 20.0, "pm10": 5.0},
    "sin campo hum (esquema)":         {k: v for k, v in base_c.items() if k != "hum"},
}
for nombre, ev in fallas.items():
    print("→", nombre); enviar(ev, ev["sensorId"], verbose=True)
print("→ texto plano (no es JSON)"); enviar(b"sensor-desconectado", "falla-texto", verbose=True)
prod.flush()

→ pm25 absurdo (850 µg/m³)
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=1 offset=50 eventType=aire.lectura sensorId=openmeteo-cams-juliaca status=published
→ temperatura imposible (-80 °C)
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=2 offset=50 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
→ pm10 menor que pm25
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=1 offset=51 eventType=aire.lectura sensorId=openmeteo-cams-juliaca status=published
→ sin campo hum (esquema)
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=2 offset=51 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
→ texto plano (no es JSON)
service=puente-openmeteo component=producer topic=ecodata-s07-lecturas partition=1 offset=52 eventType=RAW sensorId=falla-texto status=published


## 5. Consumidor `validador-ecodata` (separa esquema y rango)

In [5]:
def validar(ev):
    """Devuelve (status, detalle). Primero esquema, luego rango físico, luego coherencia."""
    if not isinstance(ev, dict): return "invalid", "no es un objeto JSON"
    tipo = ev.get("tipoEvento")
    if tipo not in CAMPOS: return "invalid", f"tipoEvento desconocido: {tipo}"
    for k, t in CAMPOS[tipo].items():
        if k not in ev: return "invalid", f"falta {k}"
        if isinstance(ev[k], bool) or not isinstance(ev[k], t): return "invalid", f"tipo inválido en {k}"
    extra = set(ev) - set(CAMPOS[tipo]) - {"tipoEvento"}
    if extra: return "invalid", f"campos no permitidos: {sorted(extra)}"
    for campo, (lo, hi) in RANGOS[tipo].items():
        if not lo <= ev[campo] <= hi: return "alerta_rango", f"{campo}={ev[campo]} fuera de [{lo}, {hi}]"
    if tipo == "aire.lectura" and ev["pm10"] < ev["pm25"]: return "alerta_coherencia", f"pm10={ev['pm10']} < pm25={ev['pm25']}"
    return "ok", ""

def validador(espera_s=6, grupo="validador-ecodata-group"):
    cons = KafkaConsumer(TOPIC, bootstrap_servers=BOOTSTRAP, group_id=grupo, auto_offset_reset="earliest",
                         consumer_timeout_ms=int(espera_s * 1000))
    filas = []
    for m in cons:
        try: ev = json.loads(m.value.decode("utf-8"))
        except Exception: ev = None
        st, det = validar(ev)
        filas.append((m.partition, m.offset, m.key.decode() if m.key else None, (ev or {}).get("tipoEvento", "—"), st, det))
        if st != "ok":
            log(service="validador-ecodata", component="consumer", topic=m.topic, partition=m.partition, offset=m.offset,
                status=st, detalle=f'"{det}"', **({"rawPayload": m.value[:40]} if ev is None else {}))
    cons.close()
    return pd.DataFrame(filas, columns=["partición", "offset", "key", "tipoEvento", "status", "detalle"])
df = validador()
print("\nResumen por status:"); print(df.status.value_counts().to_string())

service=validador-ecodata component=consumer topic=ecodata-s07-lecturas partition=2 offset=50 status=alerta_rango detalle="temp=-80.0 fuera de [-30, 35]"
service=validador-ecodata component=consumer topic=ecodata-s07-lecturas partition=2 offset=51 status=invalid detalle="falta hum"
service=validador-ecodata component=consumer topic=ecodata-s07-lecturas partition=1 offset=50 status=alerta_rango detalle="pm25=850.0 fuera de [0, 500]"
service=validador-ecodata component=consumer topic=ecodata-s07-lecturas partition=1 offset=51 status=alerta_coherencia detalle="pm10=5.0 < pm25=20.0"
service=validador-ecodata component=consumer topic=ecodata-s07-lecturas partition=1 offset=52 status=invalid detalle="no es un objeto JSON" rawPayload=b'sensor-desconectado'

Resumen por status:
status
ok                   100
alerta_rango           2
invalid                2
alerta_coherencia      1


## 6. Distribución de particiones
Una estación = una *key* = una partición: es lo correcto para conservar el orden de sus lecturas. Se muestra la distribución real y debe coincidir con Kafka UI.

In [6]:
dist = df.groupby(["partición", "key"]).size().rename("eventos").reset_index()
display(dist)
print(df.groupby("partición").offset.agg(["min", "max", "count"]))

,partición,key,eventos
0,1,falla-texto,1
1,1,openmeteo-cams-juliaca,52
2,2,openmeteo-clima-juliaca,52


           min  max  count
partición                 
1            0   52     53
2            0   51     52


## 7. Comparación con la guía (para la reflexión)
| | Guía S7 (ESP32 en Wokwi) | EcoData |
|---|---|---|
| Frecuencia | un evento cada pocos segundos | **1 por hora y variable** (la fuente actualiza cada hora) |
| Fuente | simulada | API real, pero **modelada** |
| Key | `deviceId` | `sensorId` |
| Validación | esquema + rango | esquema + rango + coherencia pm10 ≥ pm25 |
| Qué cambió | — | solo el puente; el contrato y el validador siguen igual |

### Qué capturar
1. Kafka UI → `ecodata-s07-lecturas`: 3 particiones, mensajes y la distribución.
2. Salida de la sección 3 (puente publicando) y de la 4 (fallas publicadas).
3. Salida de la sección 5: logs `status=alerta_rango`, `alerta_coherencia` e `invalid`, y el resumen por status.
4. Tabla de la sección 2 (rangos vs. valores observados) y la de la sección 6.
5. Si conectas un sensor físico (por ejemplo el ESP32), una foto del hardware: es lo que pide la rúbrica de S7.

In [ ]:
prod.close()